# <center> Cyber Attack Detection using ML

## Imports:

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix)

## Load Dataset:

In [2]:
df = pd.read_csv("train_test_network.csv")
df.head()

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor


In [3]:
print(df.shape) 
# 211,043 records (rows)
# 44 columns (features)
print(df.columns)

(211043, 44)
Index(['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service',
       'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes',
       'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query',
       'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA',
       'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed',
       'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth',
       'http_method', 'http_uri', 'http_version', 'http_request_body_len',
       'http_response_body_len', 'http_status_code', 'http_user_agent',
       'http_orig_mime_types', 'http_resp_mime_types', 'weird_name',
       'weird_addl', 'weird_notice', 'label', 'type'],
      dtype='object')


In [4]:
# Types of Attacks in the dataset:
df['type'].value_counts()

type
normal        50000
backdoor      20000
ddos          20000
dos           20000
injection     20000
password      20000
ransomware    20000
scanning      20000
xss           20000
mitm           1043
Name: count, dtype: int64

# <center> Intrusion Detection
Is this network traffic benign or malicious?

## Sample for Speed:

In [5]:
df = df.sample(30000, random_state=42) # Since TON_IoT is big.

## Separating Features & Label:

In [6]:
# Using Binary Attack detection
X = df.drop(["label","type"], axis=1)
y = df["label"]

## Encoding Categorical Features:

In [7]:
le = LabelEncoder()
for col in X.columns:
    if X[col].dtype=="object":
        X[col]=le.fit_transform(X[col].astype(str))
X.head()

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_version,http_request_body_len,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice
149315,20,40172,79,80,1,5,0.011460,155,978,10,...,0,0,0,0,0,0,0,0,0,0
66176,19,44000,304,80,1,5,1.247144,251,12183,10,...,0,0,0,0,0,0,0,0,0,0
18650,15,50068,83,80,1,0,0.000104,0,0,1,...,0,0,0,0,0,0,0,0,0,0
199593,21,52620,56,80,1,0,0.000000,0,0,11,...,0,0,0,0,0,0,0,0,0,0
60358,20,41786,57,80,1,5,0.965070,234,12183,10,...,0,0,0,0,0,0,0,0,0,0


## Spliting Data:

In [8]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.30,random_state=42)

## Scale Features:

In [9]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

## Training all Models:

In [10]:
models={"Decision Tree":DecisionTreeClassifier(),"Random Forest":RandomForestClassifier(n_estimators=100),"KNN":KNeighborsClassifier(),"SVM":SVC(),"Logistic Regression":LogisticRegression(max_iter=500),"Naive Bayes":GaussianNB()}

## Testing + Comparison:

In [11]:
results=[]
for name,model in models.items():
    print("\n",name)
    model.fit(X_train,y_train)
    preds=model.predict(X_test)
    acc=accuracy_score(y_test,preds)
    results.append([name,acc])
    print("Accuracy:",acc)
    print(classification_report(y_test,preds))


 Decision Tree
Accuracy: 0.9995555555555555
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2106
           1       1.00      1.00      1.00      6894

    accuracy                           1.00      9000
   macro avg       1.00      1.00      1.00      9000
weighted avg       1.00      1.00      1.00      9000


 Random Forest
Accuracy: 0.9996666666666667
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2106
           1       1.00      1.00      1.00      6894

    accuracy                           1.00      9000
   macro avg       1.00      1.00      1.00      9000
weighted avg       1.00      1.00      1.00      9000


 KNN
Accuracy: 0.9978888888888889
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      2106
           1       1.00      1.00      1.00      6894

    accuracy                           1.00      9000
   

## Comparing Models:

In [12]:
results_df=pd.DataFrame(results,columns=["Model","Accuracy"])
results_df.sort_values(by="Accuracy",ascending=False)

,Model,Accuracy
1,Random Forest,0.999667
0,Decision Tree,0.999556
2,KNN,0.997889
3,SVM,0.994222
4,Logistic Regression,0.992222
5,Naive Bayes,0.820778


## Choosing Best Model:

In [13]:
best_model = RandomForestClassifier(n_estimators=100)
best_model.fit(X_train,y_train)

RandomForestClassifier()

## Confusion Matrix:

In [14]:
pred=best_model.predict(X_test)
print(confusion_matrix(y_test,pred))

[[2104    2]
 [   1 6893]]


## Voting Classifier Code:

In [15]:
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,classification_report

In [16]:
# Training Voting Classifier:
voting_model = VotingClassifier(estimators=[('rf', RandomForestClassifier(n_estimators=100,random_state=42)),('gb', GradientBoostingClassifier(
random_state=42)),('dt', DecisionTreeClassifier(random_state=42))],voting='hard')

In [17]:
#  Predict:
voting_model.fit(X_train,y_train)
voting_preds = voting_model.predict(X_test)

In [18]:
# Evaluate:
print("Voting Accuracy:",accuracy_score(y_test,voting_preds))
print(classification_report(y_test,voting_preds))

Voting Accuracy: 0.9997777777777778
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2106
           1       1.00      1.00      1.00      6894

    accuracy                           1.00      9000
   macro avg       1.00      1.00      1.00      9000
weighted avg       1.00      1.00      1.00      9000



## Isolation Forest:

In [19]:
from sklearn.ensemble import IsolationForest
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [20]:
# Training:
X_normal = X_train[y_train==0]
iso = IsolationForest(n_estimators=300,contamination=0.30,random_state=42)

In [21]:
# Model Fit:
iso.fit(X_normal)

IsolationForest(contamination=0.3, n_estimators=300, random_state=42)

In [22]:
# Predict:
iso_preds = iso.predict(X_test)

In [23]:
# Convert to Binary Labels:
# Because our dataset uses attack=1 and normal=0
iso_preds = np.where(iso_preds==-1,1,0)

In [24]:
# Evaluate:
print("Isolation Forest Accuracy:",accuracy_score(y_test,iso_preds))
print(classification_report(y_test,iso_preds))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test,iso_preds))

Isolation Forest Accuracy: 0.6128888888888889
              precision    recall  f1-score   support

           0       0.34      0.70      0.46      2106
           1       0.86      0.59      0.70      6894

    accuracy                           0.61      9000
   macro avg       0.60      0.64      0.58      9000
weighted avg       0.74      0.61      0.64      9000


Confusion Matrix:

[[1464  642]
 [2842 4052]]


# <center> Attack Type Classification (Later/at the end we will do this)
Multiclass attack identification (Identify the type of attack)


# <center> Real-Time Detection Simulation

In [25]:
import time

live_data=X_test[:20]
for i,row in enumerate(live_data):
    row=row.reshape(1,-1)
    pred=best_model.predict(row)[0]
    prob=best_model.predict_proba(row)[0][1]
    print(f"\nPacket {i+1}")
    if pred==1:
        print(f"Attack Detected ({prob*100:.2f}% confidence)")
    else:
        print(f"Normal ({(1-prob)*100:.2f}% confidence)")
    time.sleep(1)


Packet 1
Normal (100.00% confidence)

Packet 2
Attack Detected (100.00% confidence)

Packet 3
Normal (100.00% confidence)

Packet 4
Attack Detected (100.00% confidence)

Packet 5
Attack Detected (100.00% confidence)

Packet 6
Attack Detected (98.00% confidence)

Packet 7
Normal (98.00% confidence)

Packet 8
Attack Detected (100.00% confidence)

Packet 9
Normal (100.00% confidence)

Packet 10
Attack Detected (100.00% confidence)

Packet 11
Attack Detected (100.00% confidence)

Packet 12
Attack Detected (100.00% confidence)

Packet 13
Attack Detected (100.00% confidence)

Packet 14
Attack Detected (100.00% confidence)

Packet 15
Attack Detected (100.00% confidence)

Packet 16
Normal (100.00% confidence)

Packet 17
Normal (100.00% confidence)

Packet 18
Attack Detected (100.00% confidence)

Packet 19
Attack Detected (100.00% confidence)

Packet 20
Attack Detected (100.00% confidence)


## Save Trained Model:

In [26]:
import joblib
joblib.dump(best_model,"cyber_model.pkl")
print("Model saved")

Model saved


## Testing single packet:

In [27]:
import requests
url="http://127.0.0.1:5000/predict"
sample = X_test[0].tolist()
response = requests.post(url,json={"features": sample})
print(response.json())

{'prediction': 'NORMAL TRAFFIC '}


## Runing Real-time stream:

In [28]:
import time
import requests
url="http://127.0.0.1:5000/predict"
for i,row in enumerate(X_test[:10]):
    response=requests.post(url,json={"features":row.tolist()})
    print(f"Packet {i+1}:",response.json())
    time.sleep(1)

Packet 1: {'prediction': 'NORMAL TRAFFIC '}
Packet 2: {'prediction': 'ATTACK DETECTED '}
Packet 3: {'prediction': 'NORMAL TRAFFIC '}
Packet 4: {'prediction': 'ATTACK DETECTED '}
Packet 5: {'prediction': 'ATTACK DETECTED '}
Packet 6: {'prediction': 'ATTACK DETECTED '}
Packet 7: {'prediction': 'NORMAL TRAFFIC '}
Packet 8: {'prediction': 'ATTACK DETECTED '}
Packet 9: {'prediction': 'NORMAL TRAFFIC '}
Packet 10: {'prediction': 'ATTACK DETECTED '}
